In [1]:
import pandas as pd
import requests
import datetime as dt
import psycopg2
import numpy as np
from sqlalchemy import create_engine

## Extract

In [2]:
#set up parameters to call api
BASE_URL="https://aviationweather.gov/api/data/metar?"
ICAO_CODES=["WSSS","WSAP","WSSL"]

In [3]:
#json that starts with [] is a list(array) and those that start with {} is a dictionary 


air_bases=[]

for code in ICAO_CODES:
    params={
    "ids":code,
    "format":'json',
    "taf":'false',
    'hour':1
    }
    response=requests.get(BASE_URL,params=params)

    if response.status_code==200:
         print (f"Data successfully retrieved for {code}")
         data=response.json()
         for dictionary in data:
             air_bases.append(dictionary)
    else:
          print (f"Error in retrieving data for {code}")

Data successfully retrieved for WSSS
Data successfully retrieved for WSAP
Data successfully retrieved for WSSL


In [4]:
print(air_bases)

[{'icaoId': 'WSSS', 'receiptTime': '2026-06-30T03:31:05.261Z', 'obsTime': 1782790200, 'reportTime': '2026-06-30T03:30:00.000Z', 'temp': 28, 'dewp': 23, 'wdir': 180, 'wspd': 6, 'visib': '6+', 'altim': 1011, 'qcField': 16, 'wxString': '-RA', 'metarType': 'METAR', 'rawOb': 'METAR WSSS 300330Z 18006KT 9999 -RA FEW018 SCT180 BKN280 28/23 Q1011 NOSIG', 'lat': 1.368, 'lon': 103.982, 'elev': 17, 'name': 'Singapore/Changi Intl, 4, SG', 'cover': 'BKN', 'clouds': [{'cover': 'FEW', 'base': 1800}, {'cover': 'SCT', 'base': 18000}, {'cover': 'BKN', 'base': 28000}], 'fltCat': 'VFR'}, {'icaoId': 'WSAP', 'receiptTime': '2026-06-30T03:02:27.453Z', 'obsTime': 1782788400, 'reportTime': '2026-06-30T03:00:00.000Z', 'temp': 29, 'dewp': 25, 'wdir': 270, 'wspd': 2, 'visib': '6+', 'altim': 1011, 'qcField': 0, 'metarType': 'METAR', 'rawOb': 'METAR WSAP 300300Z 27002KT 9999 FEW018 BKN280 29/25 Q1011', 'lat': 1.36, 'lon': 103.909, 'elev': 20, 'name': 'Singapore/Pays, 4, SG', 'cover': 'BKN', 'clouds': [{'cover': 'FE

In [5]:
records=[]

for val in air_bases:
        records.append({
            "icao_id":val['icaoId'],
            "name":val["name"],
            'report_time':val['reportTime'],
            "temperature":val["temp"],
            "dew_point":val.get('dewp'),
            "wind_direction":val.get("wdir"),
            "wind_speed":val.get("wspd"),
            'visibility':val.get('visib',0),
            "rawOb":val.get("rawOb"),
            "flight_category":val.get("fltCat"),
            "overall_cover":val.get("cover"),
        })

In [6]:
metar_df=pd.DataFrame(records)

## Transform

In [7]:
metar_df

,icao_id,name,report_time,temperature,dew_point,wind_direction,wind_speed,visibility,rawOb,flight_category,overall_cover
0,WSSS,"Singapore/Changi Intl, 4, SG",2026-06-30T03:30:00.000Z,28,23,180,6,6+,METAR WSSS 300330Z 18006KT 9999 -RA FEW018 SCT...,VFR,BKN
1,WSAP,"Singapore/Pays, 4, SG",2026-06-30T03:00:00.000Z,29,25,270,2,6+,METAR WSAP 300300Z 27002KT 9999 FEW018 BKN280 ...,VFR,BKN
2,WSSL,"Singapore/Seletar Arpt, 2, SG",2026-06-30T03:00:00.000Z,28,25,260,3,6+,METAR WSSL 300300Z 26003KT 180V320 9999 FEW015...,VFR,BKN


In [8]:
metar_df.dtypes

icao_id              str
name                 str
report_time          str
temperature        int64
dew_point          int64
wind_direction     int64
wind_speed         int64
visibility           str
rawOb                str
flight_category      str
overall_cover        str
dtype: object

In [9]:
	
#change name to country
metar_df["country"]=metar_df["name"].str.extract(r"([^/]+)")
metar_df["airport_name"] = metar_df["name"].str.extract(r"/([^,]+)")

#change airport names

metar_df["airport_name"]=metar_df["airport_name"].replace({
    "Changi Intl":"Changi",
    "Pays":"Paya Lebar",
    "Seletar Arpt":"Seletar"
    })

In [10]:
#change to SGT

metar_df["report_time"]=pd.to_datetime(metar_df["report_time"],utc=True)
metar_df["report_time"]=metar_df["report_time"].dt.tz_convert("Asia/Singapore")

#extract date and time

metar_df["date"]=metar_df["report_time"].dt.date
metar_df["time"]=metar_df["report_time"].dt.time

In [11]:
# change visibility from miles to km
metar_df["visibility"]=(
    metar_df["visibility"]
    .astype(str)
    .str.replace("+",' ')
    .astype(float)
    * 1.60934)

In [12]:
#round up values 
metar_df["visibility"]=np.ceil(metar_df["visibility"])

In [13]:
metar_df["rawOb"].unique()

<StringArray>
['METAR WSSS 300330Z 18006KT 9999 -RA FEW018 SCT180 BKN280 28/23 Q1011 NOSIG',
                  'METAR WSAP 300300Z 27002KT 9999 FEW018 BKN280 29/25 Q1011',
   'METAR WSSL 300300Z 26003KT 180V320 9999 FEW015 SCT160 BKN270 28/25 Q1011']
Length: 3, dtype: str

In [14]:
#from raw obs extract weather and convert it to a col

descriptors=r"(MI|PR|BC|DR|BL|SH|TS|FZ|VC)"
phenomena=r"(RA|DZ|BR|FG|HZ|FU|VA|SA|DU|SQ|FC|PO)"

metar_df["weather_phenomenon"]=metar_df["rawOb"].str.extract(phenomena)
metar_df["descriptors"]=metar_df["rawOb"].str.extract(descriptors)

In [15]:
#seperate string and num for wind direction

metar_df["wind_direction_var"]=metar_df["wind_direction"].eq("VRB").astype(int)

In [16]:
#if value is vrb replace it with nan
metar_df.loc[metar_df["wind_direction"]=="VRB","wind_direction"]=np.nan

In [17]:
#create unique id

metar_df["unique_id"]=(
    metar_df["rawOb"].str.split().str[1]
    +"_"+
    metar_df["rawOb"].str.split().str[2].str.replace("Z"," "))

In [18]:
#drop name,rawOb and report_time cols

metar_df.drop(columns=["name","rawOb","report_time"])

,icao_id,temperature,dew_point,wind_direction,wind_speed,visibility,flight_category,overall_cover,country,airport_name,date,time,weather_phenomenon,descriptors,wind_direction_var,unique_id
0,WSSS,28,23,180.0,6,10.0,VFR,BKN,Singapore,Changi,2026-06-30,11:30:00,RA,NaN,0,WSSS_300330
1,WSAP,29,25,270.0,2,10.0,VFR,BKN,Singapore,Paya Lebar,2026-06-30,11:00:00,SA,NaN,0,WSAP_300300
2,WSSL,28,25,260.0,3,10.0,VFR,BKN,Singapore,Seletar,2026-06-30,11:00:00,NaN,NaN,0,WSSL_300300


## Loading

In [19]:
import os
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

env_path=Path("/Users/tisya_elan/Library/Mobile Documents/com~apple~CloudDocs/Workspace/Projects/metar_weather_pipeline/etl-project/.env")

load_dotenv(env_path)

host=os.getenv("DB_HOST")
user=os.getenv("DB_USER")
password=os.getenv("DB_PASSWORD")
port=os.getenv("DB_PORT")
database=os.getenv("DB_DATABASE")

In [ ]:
#define parameters
#create engine
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

def get_connection():
    url = URL.create(
        drivername="postgresql+psycopg2",
        username=user,
        password=password,
        host=host,
        port=int(port),
        database=database,#cos password contains @ and im too lazy to change
    )
    return create_engine(url)

if __name__=='__main__':
    try:
      engine=get_connection()
      print("connection is successful")
    except Exception as ex:
       print(f"type error:{ex}")

# load dataframe
table=metar_df.to_sql(
   name='weather_etl',
   con=engine,
   if_exists='replace',
   index=False
)


print(f"no.of rows added:{table}")

connection is successful
no.of rows added:3
